In [ ]:
pip install torch numpy

In [ ]:
import numpy as np

data = np.load("/content/drive/MyDrive/something/train_data.npz")
labels = data["label"]
unique, counts = np.unique(labels, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(3256), np.int64(1): np.int64(3457), np.int64(2): np.int64(777)}


In [ ]:
# Add this to check your raw data
import numpy as np
data = np.load("/content/drive/MyDrive/something/Tripathi_dataset.npz")
X = data["data"]
y = data["label"]

print(f"Original data shape: {X.shape}")
print(f"Data type: {X.dtype}")
print(f"NaN count: {np.isnan(X).sum()}")
print(f"Inf count: {np.isinf(X).sum()}")
print(f"Finite count: {np.isfinite(X).sum()}")
print(f"Total elements: {X.size}")
print(f"Sample of data: {X[0, :5, :]}")  # First 5 timepoints of first sample

Original data shape: (8863, 1000, 3)
Data type: float64
NaN count: 1000
Inf count: 0
Finite count: 26588000
Total elements: 26589000
Sample of data: [[2.25262016e+05 2.59949744e+03 8.83961258e+01]
 [2.25347031e+05 2.59943684e+03 8.83984375e+01]
 [2.25182828e+05 2.59943031e+03 8.88912125e+01]
 [2.25314797e+05 2.59941828e+03 8.92108536e+01]
 [2.25279172e+05 2.59941349e+03 8.92880936e+01]]


In [ ]:
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import math, random, os
import sys # Import sys

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            x = augment_lightcurve(x)

        # Crucially, add .copy() to ensure contiguous array with positive strides
        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Augmentation functions from paper (minus light curve injection)
def augment_lightcurve(x):
    # Add Gaussian noise
    x += np.random.normal(0, 0.005, size=x.shape)

    # Random cyclic roll
    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)

    # Random split & swap
    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0]-1)
        x = np.concatenate((x[split:], x[:split]), axis=0)

    # Mirror flip
    if random.random() < 0.5:
        x = np.flip(x, axis=0)

    return x

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # shape (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        # CNN embedding
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Pooling + MLP
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # x: (B, T, C)
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)  # (B, T, d_model)

        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)  # (B, d_model, T)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = torch.sigmoid(logits) > 0.5
        correct += (preds == yb.bool()).sum().item()
        total += yb.size(0)
    return total_loss / len(loader.dataset), correct / total

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/Tripathi_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=15)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")

    # Use parse_known_args to ignore extra arguments from the Colab environment
    # If running in Colab, sys.argv[1:] will contain the notebook's extra arguments
    if 'ipykernel_launcher.py' in sys.argv[0]:
        # Pass only known arguments to main, excluding the script name
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()


    # Load dataset
    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']

    # Merge class 2 -> 0
    y = np.where(y == 2, 0, y)

    # Train/val/test split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # Class weights for loss
    class_counts = np.bincount(y_train)
    # Ensure there are positive and negative samples before calculating pos_weight
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        print("Warning: No positive samples found in training data. Cannot calculate pos_weight.")
        pos_weight = torch.tensor(1.0, dtype=torch.float32) # Default to 1 if no positive samples
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)


    # Balanced sampler for training
    class_sample_counts = np.bincount(y_train)
    # Avoid division by zero if a class is empty
    weights = 1. / (class_sample_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Get seq_len and in_ch from the dataset
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]

    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.cuda.amp.GradScaler()

    if args.resume:
        print(f"Loading checkpoint: {args.resume}")
        model.load_state_dict(torch.load(args.resume, map_location=device))


    # Training loop with early stopping
    best_val_loss = float('inf')
    patience = args.patience
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    # Test evaluation
    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")


if __name__ == "__main__":
    main()

Epoch 001 | Train Loss: 0.7108 | Val Loss: 0.6006 | Val Acc: 0.7652
Epoch 002 | Train Loss: 0.5501 | Val Loss: 0.5883 | Val Acc: 0.7728
Epoch 003 | Train Loss: 0.5546 | Val Loss: 0.5993 | Val Acc: 0.7682
Epoch 004 | Train Loss: 0.5737 | Val Loss: 0.5663 | Val Acc: 0.7758
Epoch 005 | Train Loss: 0.5562 | Val Loss: 0.5733 | Val Acc: 0.7758
Epoch 006 | Train Loss: 0.5407 | Val Loss: 0.5601 | Val Acc: 0.7765
Epoch 007 | Train Loss: 0.5370 | Val Loss: 0.5608 | Val Acc: 0.7765
Epoch 008 | Train Loss: 0.5439 | Val Loss: 0.5592 | Val Acc: 0.7765
Epoch 009 | Train Loss: 0.5411 | Val Loss: 0.5618 | Val Acc: 0.7765
Epoch 010 | Train Loss: 0.5384 | Val Loss: 0.5558 | Val Acc: 0.7765
Epoch 011 | Train Loss: 0.5368 | Val Loss: 0.5572 | Val Acc: 0.7765
Epoch 012 | Train Loss: 0.5381 | Val Loss: 0.5562 | Val Acc: 0.7765
Epoch 013 | Train Loss: 0.5453 | Val Loss: 0.5634 | Val Acc: 0.7765
Epoch 014 | Train Loss: 0.5536 | Val Loss: 0.5593 | Val Acc: 0.7765
Epoch 015 | Train Loss: 0.5399 | Val Loss: 0.557

In [ ]:
'''
diffusion & injection
'''
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score  # <-- added for macro F1
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            x = augment_lightcurve(x)

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Augmentation functions from paper (minus light curve injection)
def augment_lightcurve(x):
    x += np.random.normal(0, 0.005, size=x.shape)
    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)
    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0]-1)
        x = np.concatenate((x[split:], x[:split]), axis=0)
    if random.random() < 0.5:
        x = np.flip(x, axis=0)
    return x

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/Tripathi_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=25)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        print("Warning: No positive samples found in training data. Cannot calculate pos_weight.")
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    class_sample_counts = np.bincount(y_train)
    weights = 1. / (class_sample_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        print(f"Loading checkpoint: {args.resume}")
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience = args.patience
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 0.5596 | Val Loss: 0.5093 | Val Acc: 0.7743 | Val F1: 0.7640
Epoch 002 | Train Loss: 0.4691 | Val Loss: 0.4732 | Val Acc: 0.7750 | Val F1: 0.7750
Epoch 003 | Train Loss: 0.4549 | Val Loss: 0.4591 | Val Acc: 0.7961 | Val F1: 0.7955
Epoch 004 | Train Loss: 0.4351 | Val Loss: 0.4535 | Val Acc: 0.7931 | Val F1: 0.7928
Epoch 005 | Train Loss: 0.4502 | Val Loss: 0.4761 | Val Acc: 0.7803 | Val F1: 0.7800
Epoch 006 | Train Loss: 0.4704 | Val Loss: 0.4754 | Val Acc: 0.7810 | Val F1: 0.7809
Epoch 007 | Train Loss: 0.4420 | Val Loss: 0.4526 | Val Acc: 0.8074 | Val F1: 0.8039
Epoch 008 | Train Loss: 0.4406 | Val Loss: 0.4697 | Val Acc: 0.7931 | Val F1: 0.7918
Epoch 009 | Train Loss: 0.4454 | Val Loss: 0.4950 | Val Acc: 0.7547 | Val F1: 0.7546
Epoch 010 | Train Loss: 0.4592 | Val Loss: 0.4958 | Val Acc: 0.7901 | Val F1: 0.7703
Epoch 011 | Train Loss: 0.5027 | Val Loss: 0.5546 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 012 | Train Loss: 0.5471 | Val Loss: 0.5369 | Val Acc: 0.75

In [ ]:
'''
no diffusion no injection
'''# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score  # <-- added for macro F1
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            x = augment_lightcurve(x)

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Augmentation functions from paper (minus light curve injection)
def augment_lightcurve(x):
    x += np.random.normal(0, 0.005, size=x.shape)
    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)
    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0]-1)
        x = np.concatenate((x[split:], x[:split]), axis=0)
    if random.random() < 0.5:
        x = np.flip(x, axis=0)
    return x

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/merged_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=25)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        print("Warning: No positive samples found in training data. Cannot calculate pos_weight.")
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    class_sample_counts = np.bincount(y_train)
    weights = 1. / (class_sample_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        print(f"Loading checkpoint: {args.resume}")
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience = args.patience
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 1.1060 | Val Loss: 1.0733 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 002 | Train Loss: 0.9548 | Val Loss: 0.8629 | Val Acc: 0.6443 | Val F1: 0.6340
Epoch 003 | Train Loss: 0.9276 | Val Loss: 0.9588 | Val Acc: 0.6016 | Val F1: 0.5950
Epoch 004 | Train Loss: 0.9068 | Val Loss: 1.2142 | Val Acc: 0.4509 | Val F1: 0.4492
Epoch 005 | Train Loss: 0.9527 | Val Loss: 1.0469 | Val Acc: 0.4626 | Val F1: 0.4618
Epoch 006 | Train Loss: 1.0039 | Val Loss: 0.9796 | Val Acc: 0.4490 | Val F1: 0.4483
Epoch 007 | Train Loss: 1.0630 | Val Loss: 1.0790 | Val Acc: 0.3071 | Val F1: 0.2726
Epoch 008 | Train Loss: 0.9718 | Val Loss: 1.0458 | Val Acc: 0.4500 | Val F1: 0.4488
Epoch 009 | Train Loss: 0.9226 | Val Loss: 1.0549 | Val Acc: 0.4723 | Val F1: 0.4720
Epoch 010 | Train Loss: 0.9320 | Val Loss: 0.9952 | Val Acc: 0.5073 | Val F1: 0.5069
Epoch 011 | Train Loss: 0.9398 | Val Loss: 0.9845 | Val Acc: 0.5073 | Val F1: 0.5069
Epoch 012 | Train Loss: 0.9340 | Val Loss: 1.0031 | Val Acc: 0.50

In [ ]:
'''
diffusion
'''
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            x = augment_lightcurve(x)

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ====================
# Augmentation functions
# ====================
def augment_lightcurve(x):
    """
    Augments LC with:
      - Gaussian noise
      - cyclic roll
      - split & swap
      - mirror flip
    """
    x = x.copy()
    x += np.random.normal(0, 0.005, size=x.shape)

    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)

    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0]-1)
        x = np.concatenate((x[split:], x[:split]), axis=0)

    if random.random() < 0.5:
        x = np.flip(x, axis=0)

    return x

def inject_transit_gaussian(x):
    """
    Injects a Gaussian-shaped transit signal into flux channel 0.
    Returns modified x.
    """
    x = x.copy()
    T = x.shape[0]
    cadence_hours = 0.5

    # Parameter ranges (paper)
    depth = float(np.random.uniform(0.001, 0.05))  # fraction
    duration_hours = float(np.random.uniform(1.5, 12.0))
    period_days = float(np.random.uniform(0.5, 20.0))

    duration_samples = max(1, int(round(duration_hours / cadence_hours)))
    period_samples = max(1, int(round(period_days * 24.0 / cadence_hours)))
    phase_offset = np.random.randint(0, period_samples)

    # Gaussian sigma from FWHM relation: FWHM = 2.355 * sigma
    sigma = (duration_samples / 2.0) / 2.355

    positions = list(range(phase_offset, T, period_samples))
    if len(positions) == 0:
        start = np.random.randint(0, max(1, T - duration_samples))
        positions = [start]

    for pos in positions:
        center = pos + duration_samples // 2
        t = np.arange(T)
        dip = -depth * np.exp(-0.5 * ((t - center) / sigma) ** 2)
        x[:, 0] += dip

    return x

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/Tripathi_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=25)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    # Split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # --- Create injected positives from negative class ---
    injected_data = []
    injected_labels = []
    for i in range(len(X_train)):
        if y_train[i] == 0 and random.random() < 0.5:
            x_inj = inject_transit_gaussian(X_train[i])
            injected_data.append(x_inj)
            injected_labels.append(1)
    if injected_data:
        X_train = np.concatenate([X_train, np.array(injected_data)], axis=0)
        y_train = np.concatenate([y_train, np.array(injected_labels)], axis=0)

    # Class weights
    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    weights = 1. / (class_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # Datasets
    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 0.5591 | Val Loss: 0.4428 | Val Acc: 0.7750 | Val F1: 0.7413
Epoch 002 | Train Loss: 0.5055 | Val Loss: 0.4266 | Val Acc: 0.7705 | Val F1: 0.7351
Epoch 003 | Train Loss: 0.4989 | Val Loss: 0.4291 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 004 | Train Loss: 0.4997 | Val Loss: 0.4434 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 005 | Train Loss: 0.4918 | Val Loss: 0.4286 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 006 | Train Loss: 0.4995 | Val Loss: 0.4345 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 007 | Train Loss: 0.5030 | Val Loss: 0.4330 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 008 | Train Loss: 0.4979 | Val Loss: 0.4305 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 009 | Train Loss: 0.5029 | Val Loss: 0.4322 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 010 | Train Loss: 0.4967 | Val Loss: 0.4335 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 011 | Train Loss: 0.4922 | Val Loss: 0.4351 | Val Acc: 0.7765 | Val F1: 0.7433
Epoch 012 | Train Loss: 0.4916 | Val Loss: 0.4355 | Val Acc: 0.77

In [ ]:
'''
injection and diffusion
'''
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            x = augment_lightcurve(x)

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ====================
# Augmentation functions
# ====================
def augment_lightcurve(x):
    """
    Augments LC with:
      - Gaussian noise
      - cyclic roll
      - split & swap
      - mirror flip
    """
    x = x.copy()
    x += np.random.normal(0, 0.005, size=x.shape)

    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)

    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0]-1)
        x = np.concatenate((x[split:], x[:split]), axis=0)

    if random.random() < 0.5:
        x = np.flip(x, axis=0)

    return x

def inject_transit_gaussian(x):
    """
    Injects a Gaussian-shaped transit signal into flux channel 0.
    Returns modified x.
    """
    x = x.copy()
    T = x.shape[0]
    cadence_hours = 0.5

    # Parameter ranges (paper)
    depth = float(np.random.uniform(0.001, 0.05))  # fraction
    duration_hours = float(np.random.uniform(1.5, 12.0))
    period_days = float(np.random.uniform(0.5, 20.0))

    duration_samples = max(1, int(round(duration_hours / cadence_hours)))
    period_samples = max(1, int(round(period_days * 24.0 / cadence_hours)))
    phase_offset = np.random.randint(0, period_samples)

    # Gaussian sigma from FWHM relation: FWHM = 2.355 * sigma
    sigma = (duration_samples / 2.0) / 2.355

    positions = list(range(phase_offset, T, period_samples))
    if len(positions) == 0:
        start = np.random.randint(0, max(1, T - duration_samples))
        positions = [start]

    for pos in positions:
        center = pos + duration_samples // 2
        t = np.arange(T)
        dip = -depth * np.exp(-0.5 * ((t - center) / sigma) ** 2)
        x[:, 0] += dip

    return x

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/merged_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=25)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    # Split
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # --- Create injected positives from negative class ---
    injected_data = []
    injected_labels = []
    for i in range(len(X_train)):
        if y_train[i] == 0 and random.random() < 0.5:
            x_inj = inject_transit_gaussian(X_train[i])
            injected_data.append(x_inj)
            injected_labels.append(1)
    if injected_data:
        X_train = np.concatenate([X_train, np.array(injected_data)], axis=0)
        y_train = np.concatenate([y_train, np.array(injected_labels)], axis=0)

    # Class weights
    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    weights = 1. / (class_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # Datasets
    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    # Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 0.7393 | Val Loss: 0.7702 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 002 | Train Loss: 0.7367 | Val Loss: 0.7671 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 003 | Train Loss: 0.7361 | Val Loss: 0.7793 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 004 | Train Loss: 0.7373 | Val Loss: 0.7412 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 005 | Train Loss: 0.7369 | Val Loss: 0.7530 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 006 | Train Loss: 0.7361 | Val Loss: 0.7479 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 007 | Train Loss: 0.7355 | Val Loss: 0.7357 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 008 | Train Loss: 0.7370 | Val Loss: 0.7449 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 009 | Train Loss: 0.7366 | Val Loss: 0.7228 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 010 | Train Loss: 0.7363 | Val Loss: 0.7491 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 011 | Train Loss: 0.7374 | Val Loss: 0.7260 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 012 | Train Loss: 0.7363 | Val Loss: 0.7453 | Val Acc: 0.26

In [ ]:
'''

injection and diffusion part 2

'''
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score  # <-- added for macro F1
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            # augment_lightcurve now returns (x, injected_flag)
            x, injected = augment_lightcurve(x)
            if injected:
                # If we injected a transit, label it as positive (paper injects transits into non-transit LCs and labels as positive)
                y = 1

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Augmentation functions with realistic light curve injection (paper parameters)
def augment_lightcurve(x):
    """
    Augment a (T, C) numpy array.
    Returns (x_augmented, injected_flag)
    - Gaussian noise, roll, split/swap, mirror (as before)
    - With 50% probability perform a periodic box-shaped transit injection with:
        depth in [0.001, 0.05] (0.1% - 5%)
        duration in [1.5, 12] hours (converted to samples)
        period in [0.5, 20] days (converted to samples)
        random phase
      Injection applied to channel 0 only (flux).
    """
    injected = False

    # Add Gaussian noise
    x = x.copy()
    x += np.random.normal(0, 0.005, size=x.shape)

    # Random cyclic roll
    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)

    # Random split & swap
    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0] - 1)
        x = np.concatenate((x[split:], x[:split]), axis=0)

    # Mirror flip
    if random.random() < 0.5:
        x = np.flip(x, axis=0)

    # --- Light curve injection: follow paper parameters ---
    # 50% chance of injection per augmented sample
    if random.random() < 0.5:
        T = x.shape[0]       # sequence length in samples
        cadence_hours = 0.5  # TESS FFI cadence used in paper is 30 minutes -> 0.5 hours/sample

        # Transit depth fraction (normalized flux units)
        depth = float(np.random.uniform(0.001, 0.05))  # 0.1% - 5%

        # Transit duration in hours -> convert to samples
        duration_hours = float(np.random.uniform(1.5, 12.0))  # 1.5 - 12 hours
        duration_samples = max(1, int(round(duration_hours / cadence_hours)))

        # Orbital period in days -> convert to samples
        period_days = float(np.random.uniform(0.5, 20.0))  # 0.5 - 20 days
        period_samples = max(1, int(round(period_days * 24.0 / cadence_hours)))  # days * (24 / cadence_hours)

        # Random phase: pick a start offset in [0, period_samples)
        phase_offset = np.random.randint(0, period_samples)

        # We will place box dips at every period_samples starting from phase_offset,
        # across the sequence length T
        positions = list(range(phase_offset, T, period_samples))

        # Ensure at least one transit falls inside the sequence; if not, select a random start inside sequence
        if len(positions) == 0:
            start = np.random.randint(0, max(1, T - duration_samples))
            positions = [start]

        # Apply box-shaped transit dips to flux channel only (channel 0)
        for pos in positions:
            start = pos
            end = min(pos + duration_samples, T)
            if start >= T:
                continue
            # Subtract depth from flux channel (data is normalized per-channel to mean 0; depth is fraction)
            # If channel 0 scale is not exactly unit, depth is still a reasonable small perturbation.
            x[start:end, 0] = x[start:end, 0] - depth

        injected = True

    return x, injected

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/Tripathi_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=20)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        print("Warning: No positive samples found in training data. Cannot calculate pos_weight.")
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    class_sample_counts = np.bincount(y_train)
    weights = 1. / (class_sample_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        print(f"Loading checkpoint: {args.resume}")
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience = args.patience
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 0.6045 | Val Loss: 0.9384 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 002 | Train Loss: 0.5415 | Val Loss: 0.7907 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 003 | Train Loss: 0.5317 | Val Loss: 0.7971 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 004 | Train Loss: 0.5282 | Val Loss: 0.8250 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 005 | Train Loss: 0.5960 | Val Loss: 1.0925 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 006 | Train Loss: 0.6145 | Val Loss: 1.0253 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 007 | Train Loss: 0.6369 | Val Loss: 0.9907 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 008 | Train Loss: 0.6353 | Val Loss: 1.0048 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 009 | Train Loss: 0.6284 | Val Loss: 1.0634 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 010 | Train Loss: 0.6219 | Val Loss: 1.0739 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 011 | Train Loss: 0.6066 | Val Loss: 1.1298 | Val Acc: 0.4312 | Val F1: 0.3013
Epoch 012 | Train Loss: 0.6204 | Val Loss: 1.0500 | Val Acc: 0.43

In [ ]:
'''
injection only part 2
'''
# tripathi_transformer_train.py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score  # <-- added for macro F1
import math, random, os
import sys
import argparse

# ====================
# 1. Data preparation
# ====================
class LightCurveDataset(Dataset):
    def __init__(self, data, labels, augment=False):
        self.data = data
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]

        # Replace NaNs/Infs
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        # Per-channel normalization
        x = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-8)

        if self.augment:
            # augment_lightcurve now returns (x, injected_flag)
            x, injected = augment_lightcurve(x)
            if injected:
                # If we injected a transit, label it as positive (paper injects transits into non-transit LCs and labels as positive)
                y = 1

        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# Augmentation functions with realistic light curve injection (paper parameters)
def augment_lightcurve(x):
    """
    Augment a (T, C) numpy array.
    Returns (x_augmented, injected_flag)
    - Gaussian noise, roll, split/swap, mirror (as before)
    - With 50% probability perform a periodic box-shaped transit injection with:
        depth in [0.001, 0.05] (0.1% - 5%)
        duration in [1.5, 12] hours (converted to samples)
        period in [0.5, 20] days (converted to samples)
        random phase
      Injection applied to channel 0 only (flux).
    """
    injected = False

    # Add Gaussian noise
    x = x.copy()
    x += np.random.normal(0, 0.005, size=x.shape)

    # Random cyclic roll
    if random.random() < 0.5:
        shift = np.random.randint(0, x.shape[0])
        x = np.roll(x, shift, axis=0)

    # Random split & swap
    if random.random() < 0.5:
        split = np.random.randint(1, x.shape[0] - 1)
        x = np.concatenate((x[split:], x[:split]), axis=0)

    # Mirror flip
    if random.random() < 0.5:
        x = np.flip(x, axis=0)

    # --- Light curve injection: follow paper parameters ---
    # 50% chance of injection per augmented sample
    if random.random() < 0.5:
        T = x.shape[0]       # sequence length in samples
        cadence_hours = 0.5  # TESS FFI cadence used in paper is 30 minutes -> 0.5 hours/sample

        # Transit depth fraction (normalized flux units)
        depth = float(np.random.uniform(0.001, 0.05))  # 0.1% - 5%

        # Transit duration in hours -> convert to samples
        duration_hours = float(np.random.uniform(1.5, 12.0))  # 1.5 - 12 hours
        duration_samples = max(1, int(round(duration_hours / cadence_hours)))

        # Orbital period in days -> convert to samples
        period_days = float(np.random.uniform(0.5, 20.0))  # 0.5 - 20 days
        period_samples = max(1, int(round(period_days * 24.0 / cadence_hours)))  # days * (24 / cadence_hours)

        # Random phase: pick a start offset in [0, period_samples)
        phase_offset = np.random.randint(0, period_samples)

        # We will place box dips at every period_samples starting from phase_offset,
        # across the sequence length T
        positions = list(range(phase_offset, T, period_samples))

        # Ensure at least one transit falls inside the sequence; if not, select a random start inside sequence
        if len(positions) == 0:
            start = np.random.randint(0, max(1, T - duration_samples))
            positions = [start]

        # Apply box-shaped transit dips to flux channel only (channel 0)
        for pos in positions:
            start = pos
            end = min(pos + duration_samples, T)
            if start >= T:
                continue
            # Subtract depth from flux channel (data is normalized per-channel to mean 0; depth is fraction)
            # If channel 0 scale is not exactly unit, depth is still a reasonable small perturbation.
            x[start:end, 0] = x[start:end, 0] - depth

        injected = True

    return x, injected

# ====================
# 2. Positional Encoding
# ====================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# ====================
# 3. CNN + Transformer Model
# ====================
class CNNTransformer(nn.Module):
    def __init__(self, seq_len=1000, in_ch=3, d_model=128, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, 64, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(64, d_model, kernel_size=5, padding=2)
        self.act = nn.GELU()
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(d_model, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = x.permute(0, 2, 1)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.permute(0, 2, 1)
        x = self.pool(x).squeeze(-1)
        x = self.act(self.fc1(x))
        return self.fc2(x).squeeze(-1)

# ====================
# 4. Training utils
# ====================
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        preds = (torch.sigmoid(logits) > 0.5).long()
        correct += (preds.view(-1) == yb.long()).sum().item()
        total += yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
    acc = correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader.dataset), acc, f1

# ====================
# 5. Main
# ====================
def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", default="/content/drive/MyDrive/something/merged_dataset.npz", help="Path to the .npz data file")
    p.add_argument("--batch", type=int, default=64)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--max_epochs", type=int, default=200)
    p.add_argument("--patience", type=int, default=20)
    p.add_argument("--resume", default=None, help="path to .pt checkpoint")
    if 'ipykernel_launcher.py' in sys.argv[0]:
        args, unknown = p.parse_known_args(sys.argv[1:])
    else:
        args, unknown = p.parse_known_args()

    arr = np.load(args.data, allow_pickle=True)
    X = arr['data']
    y = arr['label']
    y = np.where(y == 2, 0, y)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    class_counts = np.bincount(y_train)
    if class_counts.shape[0] < 2 or class_counts[1] == 0:
        print("Warning: No positive samples found in training data. Cannot calculate pos_weight.")
        pos_weight = torch.tensor(1.0, dtype=torch.float32)
    else:
        pos_weight = torch.tensor(class_counts[0] / class_counts[1], dtype=torch.float32)

    class_sample_counts = np.bincount(y_train)
    weights = 1. / (class_sample_counts + 1e-9)
    sample_weights = weights[y_train]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_ds = LightCurveDataset(X_train, y_train, augment=True)
    val_ds = LightCurveDataset(X_val, y_val, augment=False)
    test_ds = LightCurveDataset(X_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=args.batch, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch, shuffle=False, num_workers=2, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sample_x, _ = train_ds[0]
    seq_len = sample_x.shape[0]
    in_ch = sample_x.shape[1]
    model = CNNTransformer(seq_len=seq_len, in_ch=in_ch).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.8, patience=5)
    scaler = torch.amp.GradScaler('cuda')

    if args.resume:
        print(f"Loading checkpoint: {args.resume}")
        model.load_state_dict(torch.load(args.resume, map_location=device))

    best_val_loss = float('inf')
    patience = args.patience
    patience_counter = 0
    for epoch in range(1, args.max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if os.path.exists("best_model.pt"):
        model.load_state_dict(torch.load("best_model.pt"))
        test_loss, test_acc, test_f1 = eval_epoch(model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | Test F1: {test_f1:.4f}")
    else:
        print("No best_model.pt found to evaluate test set.")

if __name__ == "__main__":
    main()


Epoch 001 | Train Loss: 0.8224 | Val Loss: 1.6293 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 002 | Train Loss: 0.7739 | Val Loss: 1.6701 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 003 | Train Loss: 0.7920 | Val Loss: 1.7953 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 004 | Train Loss: 0.7832 | Val Loss: 1.6880 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 005 | Train Loss: 0.7944 | Val Loss: 1.7792 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 006 | Train Loss: 0.7671 | Val Loss: 1.8072 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 007 | Train Loss: 0.8119 | Val Loss: 1.8643 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 008 | Train Loss: 0.7921 | Val Loss: 1.6531 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 009 | Train Loss: 0.7947 | Val Loss: 1.7337 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 010 | Train Loss: 0.7876 | Val Loss: 1.7101 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 011 | Train Loss: 0.8115 | Val Loss: 1.7191 | Val Acc: 0.2653 | Val F1: 0.2097
Epoch 012 | Train Loss: 0.7878 | Val Loss: 1.7081 | Val Acc: 0.26